In [ ]:
import sqlite3
import pandas as pd
import os
import re
import numpy as np
from sklearn import cross_validation, metrics
from sklearn.metrics import accuracy_score, roc_curve, auc,  precision_recall_curve, average_precision_score
from sklearn.grid_search import GridSearchCV
from nltk.tokenize import sent_tokenize, word_tokenize, StanfordTokenizer
from nltk.corpus import stopwords
#from bs4 import BeautifulSoup
import matplotlib.pylab as plt
%matplotlib inline
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 12, 4
import string
import seaborn as sns
sns.set_style("whitegrid")
os.chdir('/Users/RayLJazz/Dropbox/iidata')

In [ ]:
ffclean = pd.read_csv('cleaned_food.csv')

In [ ]:
ffclean.shape, ffclean.columns

In [ ]:
Class = []
def identify_class(row):
    if (row['ingredients_that_may_be_from_palm_oil_n']==np.nan):
        Class.append(np.nan)
    elif (row['ingredients_that_may_be_from_palm_oil_n']==0):
        Class.append(int(0))
    else:
        Class.append(int(1))

In [ ]:
ffclean_class = ffclean.apply(identify_class,axis=1)

In [ ]:
Class_df = pd.Series(data=Class)
ffclean['palm_Class'] = Class_df
ffclean['palm_Class'].value_counts()

In [ ]:
Class_char = pd.Categorical.from_codes(ffclean['palm_Class'], ['No', 'Yes'])
ffclean['Class_char'] = Class_char
ffclean_NA=ffclean.replace('NA', np.nan)
ffclean_NA=ffclean.replace('NaN', np.nan)
#ffclean_rna = ffclean_NA.dropna(axis=0, how='any')
ffclean_rna = ffclean

In [ ]:
ffclean_rna.shape,ffclean_rna.columns

In [ ]:
ffclean_rna.head(n=10)
target = 'palm_Class'
Predictors = [x for x in ffclean.columns if x not in ['ingredients_that_may_be_from_palm_oil_n','quan_unit','Class_char','palm_Class','packaging_cleaned']]
Predictors 

In [ ]:
ffclean_rna.shape

In [ ]:
train_x, test_x, train_y, test_y = cross_validation.train_test_split(ffclean_rna[Predictors], ffclean_rna['palm_Class'], test_size=0.3,random_state=1,stratify=ffclean_rna['palm_Class'])

In [ ]:
train_y.value_counts(),  test_y.value_counts()

In [ ]:
train_x.shape, test_x.shape, train_y.shape, test_y.shape

## Extreme Gradient Boosting

### Step 1- Find the number of estimators for a high learning rate

In [ ]:
import xgboost as xgb
from xgboost.sklearn import XGBClassifier

In [ ]:
train = train_x
train['Class'] = train_y
test = test_x
test['Class'] = test_y
target = 'Class'

In [ ]:
def modelfit(alg, dtrain, dtest, predictors,target, useTrainCV=True, cv_folds=5, early_stopping_round=50):
    
    if useTrainCV:
        xgb_param = alg.get_xgb_params()
        xgtrain = xgb.DMatrix(dtrain[predictors].values, label=dtrain[target].values,missing=np.nan)
        xgtest = xgb.DMatrix(dtest[predictors].values,missing=np.nan)
        xgb_param_forcv = xgb_param
        #xgb_param_forcv['num_class'] = 5
        cvresult = xgb.cv(xgb_param_forcv, xgtrain, num_boost_round=alg.get_params()['n_estimators'], nfold=cv_folds,
             early_stopping_rounds=early_stopping_round, show_progress=True)
        alg.set_params(n_estimators=cvresult.shape[0])
    
    #Fit the algorithm on the data
    alg.fit(dtrain[predictors], dtrain[target],eval_metric='error')
    
    #Predict training set:
    dtrain_predictions = alg.predict(dtrain[predictors])
    dtrain_predprob = alg.predict_proba(dtrain[predictors])[:,1]
    return alg, dtrain_predictions, dtrain_predprob
   

In [ ]:
xgb1 = XGBClassifier(
        learning_rate =0.1,
        n_estimators=1000,
        max_depth=5,
        min_child_weight=1,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        objective= 'binary:logistic',
        nthread=4,
        scale_pos_weight=1,
        seed=27)
xgb1, dtrain_predictions, dtrain_predprob = modelfit(xgb1, train, test, Predictors,target)

In [ ]:
def get_performance(xgbmodel,dtrain,dtest,predictors,dtrain_predictions):
    print ("\nModel Report")
    print ("Train accuracy : %.4g" % metrics.accuracy_score(dtrain[target].values, dtrain_predictions))
    test_result = xgbmodel.predict(dtest[predictors])
    dtest_predprob = xgbmodel.predict_proba(dtest[predictors])[:,1]
    print ("Test accuracy : %.4g" % metrics.accuracy_score(dtest[target].values, test_result))
    print ("AUC Score (Train): %f" % metrics.roc_auc_score(dtrain[target], dtrain_predprob))
    print ("AUC Score (Test): %f" % metrics.roc_auc_score(dtest[target], dtest_predprob))
    print (pd.crosstab(dtest[target].values, test_result, rownames=['actual'], colnames=['preds']))
    feat_imp = pd.Series(xgbmodel.booster().get_fscore()).sort_values(ascending=False)
    feat_imp.plot(kind='barh', title='Feature Importances').invert_yaxis()
    plt.ylabel('Feature Importance Score')

In [ ]:
get_performance(xgb1,train,test,Predictors,dtrain_predictions)